# 实验五（课堂演示）：语义通信 vs 传统通信

**适用课程**：未来媒体互联网
**演示时长**：约 10 分钟
**运行环境**：Kaggle Notebook（CPU，PyTorch 预装）

## 演示目标

对比传统方案和语义通信方案在不同信噪比下的重建效果。
核心震撼点：极低 SNR（-10dB）下传统方案完全失效，语义方案仍能保留数字的语义信息。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports OK')

In [ ]:
# Load MNIST data
transform = transforms.ToTensor()
train_set = torchvision.datasets.MNIST(
    root='./mnist_data', train=True, download=True, transform=transform
)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)

# Show samples
samples, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    axes[i].imshow(samples[i][0], cmap='gray')
    axes[i].set_title(str(labels[i].item()))
    axes[i].axis('off')
plt.suptitle('MNIST Samples')
plt.tight_layout()
plt.show()
print(f'Training set: {len(train_set)} images')

In [ ]:
LATENT_DIM = 16

class SemanticAutoencoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )

    def add_channel_noise(self, z, snr_db):
        if snr_db is None:
            return z
        signal_power = z.pow(2).mean(dim=1, keepdim=True)
        snr_linear = 10 ** (snr_db / 10.0)
        noise_power = signal_power / snr_linear
        noise = torch.randn_like(z) * torch.sqrt(noise_power)
        return z + noise

    def forward(self, x, snr_db=10):
        z = self.encoder(x.view(x.size(0), -1))
        z_noisy = self.add_channel_noise(z, snr_db)
        return self.decoder(z_noisy).view(-1, 1, 28, 28)

model = SemanticAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train (about 2 minutes on CPU)
EPOCHS = 5
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        output = model(data, snr_db=10)
        loss = criterion(output, data)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}')
print('Training done')

In [ ]:
# Select one image for clear comparison
model.eval()
demo_loader = torch.utils.data.DataLoader(train_set, batch_size=10, shuffle=True)
test_images, test_labels = next(iter(demo_loader))
test_images = test_images.to(device)

# Pick a clear digit to demonstrate
idx = 0
img = test_images[idx:idx+1]
label = test_labels[idx].item()
print(f'Selected digit: {label}')

# Traditional: add noise directly to pixels
def traditional_transmit(img, snr_db):
    if snr_db >= 20:
        return img.clone()
    sig_pow = img.pow(2).mean()
    snr_lin = 10 ** (snr_db / 10.0)
    if snr_lin == 0:
        return img.clone()
    noise_pow = sig_pow / snr_lin
    noise = torch.randn_like(img) * torch.sqrt(noise_pow)
    return torch.clamp(img + noise, 0, 1)

# Compare at different SNR levels
snr_levels = [20, 10, 0, -5, -10]

fig, axes = plt.subplots(2, len(snr_levels) + 1, figsize=(16, 7))

# Original image
axes[0, 0].imshow(img.cpu().squeeze(), cmap='gray')
axes[0, 0].set_title(f'Original\nDigit: {label}', fontsize=10, fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for j, snr in enumerate(snr_levels):
    # Traditional
    trad = traditional_transmit(img, snr)
    axes[0, j+1].imshow(trad.cpu().squeeze(), cmap='gray')
    axes[0, j+1].set_title(f'Traditional\nSNR={snr}dB', fontsize=9)
    axes[0, j+1].axis('off')

    # Semantic
    with torch.no_grad():
        sem = model(img, snr_db=snr)
    axes[1, j+1].imshow(sem.cpu().squeeze(), cmap='gray')
    axes[1, j+1].set_title(f'Semantic\nSNR={snr}dB', fontsize=9)
    axes[1, j+1].axis('off')

plt.suptitle(f'Semantic vs Traditional: Digit {label} at Different SNR\n(Top: Traditional | Bottom: Semantic)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key observation:')
print(f'At SNR=-10dB, traditional is pure noise (digit {label} unrecognizable)')
print(f'But semantic communication still preserves the shape of {label}')
print('Semantic comm trades pixel accuracy for meaning preservation.')

## 课堂演示流程（10 分钟）

1. 运行 Cell 1-2：展示 MNIST 样本
2. 运行 Cell 3-4：训练模型（约 2 分钟），期间讲解自编码器概念
3. 运行 Cell 5：展示对比结果
4. 重点观察 SNR=-10dB 列：传统方案全雪花，语义方案还能看出数字

## 课堂互动

1. 语义通信牺牲了什么，换来了什么？
2. 为什么这种 trade-off 适合 6G 场景？
3. 如果传输的不是 MNIST 数字而是人脸，结果会一样吗？